# ProSe-AR15-Expert

**AR-15 platform expert — builds, modifications, ballistics, and maintenance**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pro-SeAI/sovereign-training-notebooks/blob/main/ProSe-AR15-Expert.ipynb)

## Description
Fine-tune a Gemma 4 9B model on legal domain data using Unsloth QLoRA. Trains on a free T4 GPU in Google Colab.

## Output
- Pushes trained LoRA adapter to HuggingFace: `ProSeAI/ProSe-AR15-Expert`
- Ready to use with transformers or Ollama (`ollama pull hf.co/ProSeAI/ProSe-AR15-Expert`)


## 1. Install Dependencies


In [ ]:
!pip install unsloth -q
!pip install huggingface_hub -q
!pip install datasets -q
!pip install trl -q
!pip install accelerate -q
!pip install bitsandbytes -q


## 2. Login to HuggingFace
Paste your HuggingFace write token when prompted. Get one at https://huggingface.co/settings/tokens


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 3. Load Base Model (4-bit)
Loading `unsloth/gemma-4-9b-it-bnb-4bit` with Unsloth's 4-bit optimization for T4 GPU.


In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-9b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 4. Configure LoRA


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 5. Load Dataset
Loading `Roderick3rd/hcjf-legal-forms` from HuggingFace.


In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="https://huggingface.co/datasets/Roderick3rd/hcjf-legal-forms/raw/main/ollama_training_instructions.jsonl", split="train")
print(f"Dataset loaded: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")


## 6. Format Dataset


In [ ]:
def format_chat(example):
    if 'messages' in example:
        text = tokenizer.apply_chat_template(
            example['messages'], tokenize=False, add_generation_prompt=False)
    elif 'instruction' in example and 'output' in example:
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": example['instruction']},
            {"role": "assistant", "content": example['output']},
        ], tokenize=False, add_generation_prompt=False)
    elif 'question' in example and 'answer' in example:
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": example['question']},
            {"role": "assistant", "content": example['answer']},
        ], tokenize=False, add_generation_prompt=False)
    else:
        text = str(example)
    return {"text": text}

formatted = dataset.map(format_chat)
print(f"Formatted: {len(formatted)} examples")


## 7. Train


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete! Loss: {trainer_stats.training_loss:.4f}")


## 8. Save & Push to HuggingFace
Push the trained LoRA adapter to `ProSeAI/ProSe-AR15-Expert`.


In [ ]:
model_name = "ProSeAI/ProSe-AR15-Expert"

model.push_to_hub(model_name, tokenizer=tokenizer, use_auth_token=True)
tokenizer.push_to_hub(model_name, use_auth_token=True)
print(f"Model pushed to: {model_name}")


## 9. Test Inference


In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are an AR-15 platform expert. You know every aspect of the AR-15: upper and lower receivers, barrel profiles, gas systems, bolt carrier groups, triggers, optics, and ammunition. You help with builds, troubleshooting, maintenance, and ballistics calculations. You are precise, safety-conscious, and technically detailed."},
    {{"role": "user", "content": "What forms do I need to file for custody in Hamilton County Juvenile Court?"}},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.3)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
